# SAM-Audio Large · Colab 업로드용
**이 ipynb 파일 하나만 Colab에 업로드하세요. 별도 YAML이나 모델 파일은 필요 없습니다.**

1. 런타임 → 런타임 유형 변경 → **A100 GPU**(가능하면 고용량 RAM)를 선택합니다. Large는 A100 GPU와 고용량 시스템 RAM을 권장합니다. T4는 이 기본 BF16 설정을 지원하지 않습니다.
2. 왼쪽 열쇠(Secrets)에 `HF_TOKEN`과 `WANDB_API_KEY`를 등록하고 각각 노트북 액세스를 켭니다. HF_TOKEN은 [SAM-Audio Large](https://huggingface.co/facebook/sam-audio-large) 접근 권한이 있는 계정의 읽기 토큰입니다. Small 승인만 받았다면 Large 접근 가능 여부도 확인하세요.
3. 런타임 → **모두 실행**을 누릅니다. Google 인증 창에서는 버킷 권한이 있는 계정을 선택하세요.
4. 완료되면 W&B에 SDR이 기록되고 결과 ZIP 다운로드가 시작됩니다.

GCP: `ensemblebanchmark` / `gs://ensemble_banchmark/prepared`. W&B: `orangekiwi-ajou-university/ensemble`. 변경이 필요하면 2번 셀의 `CONFIG_YAML`을 수정하세요.

추가 학습 없이 전체 validation을 평가합니다. 기존 baseline과 같은 SDR 평가기를 사용합니다. 모델·코드 revision, 텍스트 프롬프트, 10초 구간과 1초 문맥, 16 ODE 설정을 고정했습니다. 로컬과 같은 텍스트 전용 경로를 사용하며 GPU 차이로 수치가 달라질 수 있습니다.

**실제 Colab GPU 실행은 아직 검증하지 않았으며 10시간 이내 완료를 보장하지 않습니다.** 런타임이 삭제되면 `/content`의 중간 결과도 사라집니다. 현재 노트북은 런타임 간 재개 기능을 제공하지 않습니다.


## 1번 셀 · GPU 확인 및 필요한 패키지만 설치
기존 PyTorch·torchaudio·CUDA를 확인하고 버전을 유지합니다. FFmpeg는 없을 때만 설치합니다. 추가 패키지도 호환되는 버전이 이미 있으면 재설치하지 않습니다. 기존 GPU 환경 검사에 실패하면 새 A100 런타임에서 다시 실행하세요.


In [ ]:
# Set before importing transformers or sam_audio: this evaluation uses PyTorch only.
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'

import subprocess


def run_live(command):
    # Forward child-process output through Python so Colab displays it immediately.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


print('[1/5] GPU 확인 시작', flush=True)
run_live(['nvidia-smi'])
REPOSITORY = "c-peace/ensemble_banchmark_pipeline"
REF = "b7cbf44476ed2017c2155cb4e1a233f3faac4e5e"

import os
import re
import subprocess
import sys
import tempfile
import shutil
from importlib import metadata
from pathlib import Path



def preserved_gpu_requirements():
    # Prevent transitive dependencies from replacing Colab's working GPU stack.
    requirements = []
    for distribution in metadata.distributions():
        name = distribution.metadata.get('Name', '').lower().replace('_', '-')
        if name in {'torch', 'torchaudio', 'torchvision', 'triton'} or name.startswith('nvidia-'):
            requirements.append(f'{name}=={distribution.version}')
    return sorted(set(requirements))


# Validate in a subprocess so installing other packages cannot leave stale imports.
print('Colab 기본 PyTorch·CUDA 호환성 확인', flush=True)
run_live([sys.executable, '-c', """
import torch, torchaudio
if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('BF16 지원 GPU가 필요합니다. A100 런타임을 선택하세요.')
x = torch.zeros(1, 4410, device='cuda')
torchaudio.functional.resample(x, 44100, 48000)
torch.cuda.synchronize()
print('기존 환경 사용:', torch.__version__, torchaudio.__version__, torch.cuda.get_device_name())
"""])

def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("REPOSITORY는 GitHub의 owner/repository 형식입니다.")
if not REF or REF.startswith("-") or not re.fullmatch(r"[A-Za-z0-9_./-]+", REF):
    raise ValueError("REF에 브랜치, 태그 또는 커밋을 입력하세요.")
print('[2/5] GitHub 평가 코드 다운로드', flush=True)
CODE_ROOT = Path("/content/ensemble-code")
url = f"https://github.com/{REPOSITORY}.git"
with tempfile.TemporaryDirectory() as temporary:
    askpass = Path(temporary) / "askpass.py"
    askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('GITHUB_TOKEN', ''))\n")
    askpass.chmod(0o700)
    git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(askpass),
                   GITHUB_TOKEN=secret("GITHUB_TOKEN") or "")

    def git(*args):
        result = subprocess.run(["git", *args], env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("GitHub 다운로드 실패: 저장소·REF·토큰 권한을 확인하세요.")
        return result.stdout.strip()

    if not CODE_ROOT.exists():
        # 실패한 다운로드가 다음 실행에 남지 않도록 임시 폴더에서 준비합니다.
        checkout = Path(temporary) / "checkout"
        git("init", "--quiet", str(checkout))
        git("-C", str(checkout), "remote", "add", "origin", url)
        git("-C", str(checkout), "fetch", "--quiet", "--depth", "1", "origin", REF)
        git("-C", str(checkout), "checkout", "--quiet", "--detach", "FETCH_HEAD")
        import shutil
        shutil.move(str(checkout), str(CODE_ROOT))
    else:
        if git("-C", str(CODE_ROOT), "remote", "get-url", "origin") != url:
            raise ValueError("다른 저장소가 이미 있습니다. 새 Colab 런타임에서 실행하세요.")
        if git("-C", str(CODE_ROOT), "status", "--porcelain"):
            raise ValueError("가져온 코드가 수정되었습니다. 변경을 저장하고 새 런타임에서 실행하세요.")
        git("-C", str(CODE_ROOT), "fetch", "--quiet", "--depth", "1", "origin", REF)
        if git("-C", str(CODE_ROOT), "rev-parse", "HEAD") != git("-C", str(CODE_ROOT), "rev-parse", "FETCH_HEAD"):
            raise ValueError("GitHub 코드가 변경되었습니다. 새 런타임에서 다시 시작하세요.")
    CODE_COMMIT = git("-C", str(CODE_ROOT), "rev-parse", "HEAD")
os.chdir(CODE_ROOT)
SAM_COMMIT = "bb4c6999d2677c7402360e426afc01ddfad6dce0"
print('[3/5] FFmpeg 확인', flush=True)
if shutil.which('ffmpeg'):
    print('이미 설치되어 있어 건너뜁니다.', flush=True)
else:
    run_live(['apt-get', 'update', '-qq'])
    run_live(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
# Use the text-only path already exercised locally; unused vision/rankers are excluded.
print('[4/5] 부족하거나 호환되지 않는 SAM-Audio 의존성만 설치', flush=True)
constraints = Path('/content/colab-gpu-constraints.txt')
constraints.write_text('\n'.join(preserved_gpu_requirements()) + '\n')
run_live([sys.executable, '-u', '-m', 'pip', 'install', '--progress-bar', 'off',
    '--constraint', str(constraints), '.[colab]',
    'transformers>=4.54,<5', 'torchdiffeq>=0.2.5,<0.3',
    'dacvae @ git+https://github.com/facebookresearch/dacvae.git@414c20785fc3a28373073ea8ef7a1316eeeaca6e'])
print('[5/5] SAM-Audio 코드 준비', flush=True)
SAM_ROOT = Path('/content/sam-audio-source')
if not SAM_ROOT.exists():
    run_live(['git', 'clone', '--progress', 'https://github.com/facebookresearch/sam-audio.git', str(SAM_ROOT)])
run_live(['git', '-C', str(SAM_ROOT), 'checkout', '--detach', SAM_COMMIT])
from pathlib import Path
root=SAM_ROOT/'sam_audio'
p=root/'model'/'__init__.py'
s=p.read_text().replace('from .judge import *  # noqa','')
p.write_text(s)
p=root/'model'/'config.py'
s=p.read_text().replace('from core.audio_visual_encoder.config import TransformerConfig as PEAVTransformerConfig','PEAVTransformerConfig = None  # Text-only runner never constructs Judge configs.')
p.write_text(s)
p=root/'model'/'model.py'
s=p.read_text().replace('from core.audio_visual_encoder import PEAudioFrame, PEAudioFrameTransform', '''# Local text-only execution: optional span prediction is disabled.
PEAudioFrame = PEAudioFrameTransform = None''')
s=s.replace('from sam_audio.model.vision_encoder import PerceptionEncoder', '''class PerceptionEncoder(torch.nn.Module):
    # With masked_video=None, official _get_video_features uses only .dim.
    def __init__(self, cfg):
        super().__init__()
        self.dim = cfg.dim
    def forward(self, *args, **kwargs):
        raise RuntimeError("This runner supports text prompts only")''')
s=s.replace('from sam_audio.ranking import create_ranker', '''def create_ranker(config):
    if config is not None:
        raise RuntimeError("Reranking must be disabled in the local text-only runner")
    return None''')
p.write_text(s)
p=root/'processor.py'
s=p.read_text().replace('from torchcodec.decoders import AudioDecoder, VideoDecoder', '''class UnusedVideoDecoder:
    def __init__(self, *args, **kwargs):
        raise RuntimeError("Use preloaded audio tensors in this text-only runner")
AudioDecoder = VideoDecoder = UnusedVideoDecoder''')
p.write_text(s)

sys.path.insert(0, str(SAM_ROOT))
print('코드 준비 완료:', CODE_COMMIT)

import torch
if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('BF16 지원 GPU가 필요합니다. Large 평가에는 A100을 권장합니다.')
for required in ('HF_TOKEN', 'WANDB_API_KEY'):
    if not secret(required):
        raise ValueError(f'Colab Secrets에 {required}를 등록하고 노트북 액세스를 켜세요.')
print('GPU:', torch.cuda.get_device_name())


## 2번 셀 · validation만 다운로드
권한을 받은 Google 계정으로 인증하세요. 원본 음원은 변경하지 않습니다.

In [ ]:
from google.colab import auth
from ensemble_benchmark.cloud import download_data
from ensemble_benchmark.config import load_config

# YAML is embedded: no separate file upload is needed.
CONFIG_YAML = '# 추가 학습 없이 전체 validation 평가. max_scenes: 1이면 일부 장면 진단 평가.\n# chunk_seconds는 처리 단위이며, 전체 음원의 평가 길이를 줄이지 않습니다.\nschema_version: 1\nexperiment:\n  name: sam-audio-large-pretrained\n  wandb_project: ensemble\n  wandb_entity: orangekiwi-ajou-university\n  wandb_mode: online\ndata:\n  gcp_project: ensemblebanchmark\n  gcs_uri: gs://ensemble_banchmark/prepared\n  smoke: false\nevaluation:\n  enabled: true\n  allow_partial: false\n  max_seconds: null\n  include_metrics: false\nparameters:\n  model_id: facebook/sam-audio-large\n  model_revision: 5f2cd3a9471a08c7282c06036be6893e18de8b70\n  dtype: bfloat16\n  chunk_seconds: 10.0\n  context_seconds: 1.0\n  seed: 42\n  max_scenes: null\n  prompts:\n    Violin: violin playing\n    Viola: viola playing\n    Cello: cello playing\n    Bass: double bass playing\n    Flute: flute playing\n    Clarinet: clarinet playing\n    Oboe: oboe playing\n    Bassoon: bassoon playing\n    Horn: french horn playing\n    Trumpet: trumpet playing\n    Trombone: trombone playing\n    Tuba: tuba playing\n    Harp: harp playing\n    Timpani: timpani playing\n    untunedpercussion: unpitched percussion playing\n'
config_path = Path('/content/sam-audio-settings.yaml')
config_path.write_text(CONFIG_YAML)
config = load_config(config_path)
auth.authenticate_user(project_id=config['data']['gcp_project'])
DATA_ROOT = download_data(config['data']['gcs_uri'], '/content/ensemble-data',
    project=config['data']['gcp_project'], splits=('validation',))
VALID_ROOT = DATA_ROOT / 'validation'
print('Validation:', VALID_ROOT)

## 3번 셀 · 새 W&B 평가 실험 시작
이 셀을 다시 실행하면 새 run이 만들어집니다. 프롬프트는 목표 악기 이름만 사용합니다. 정답 음원의 시간 정보나 내용으로 프롬프트를 고르지 않습니다.
`Bass`는 double bass, `Horn`은 french horn, `untunedpercussion`은 unpitched percussion으로 구분했습니다. 프롬프트 성능은 아직 검증되지 않았으며 YAML에서 수정할 수 있습니다.

In [ ]:
import json
import uuid
import yaml
import math
import copy
import wandb

p = config['parameters']
if p['model_id'] != 'facebook/sam-audio-large':
    raise ValueError('이 노트북은 sam-audio-large 평가용입니다.')
for key in ('chunk_seconds', 'context_seconds'):
    if not isinstance(p[key], (int, float)) or not math.isfinite(p[key]) or p[key] < 0:
        raise ValueError(f'{key} must be finite and nonnegative')
if p['chunk_seconds'] <= 0 or p['context_seconds'] >= p['chunk_seconds']:
    raise ValueError('0 <= context_seconds < chunk_seconds 이어야 합니다.')
if p['dtype'] not in ('bfloat16', 'float32'):
    raise ValueError('dtype은 bfloat16 또는 float32입니다.')
if p['max_scenes'] is not None and (type(p['max_scenes']) is not int or p['max_scenes'] < 1):
    raise ValueError('max_scenes는 null 또는 양의 정수입니다.')
if type(p['seed']) is not int or not 0 <= p['seed'] < 2**32:
    raise ValueError('seed must be an integer in [0, 2**32)')
if not isinstance(p['prompts'], dict) or not all(isinstance(v, str) and v.strip() for v in p['prompts'].values()):
    raise ValueError('악기별 텍스트 프롬프트를 입력하세요.')
if (config['data']['smoke'] or not config['evaluation']['enabled']
        or config['evaluation']['max_seconds'] is not None or config['evaluation']['include_metrics']):
    raise ValueError('이 노트북은 길이 제한 없는 SDR 평가입니다. 장면 제한은 parameters.max_scenes를 사용하세요.')
experiment = config['experiment']
if wandb.run is not None:
    wandb.run.finish()
if experiment['wandb_mode'] == 'online':
    api_key = secret('WANDB_API_KEY')
    if not api_key:
        raise ValueError('Colab Secrets에 WANDB_API_KEY를 등록하고 노트북 액세스를 켜세요.')
    wandb.login(key=api_key)
    del api_key
RUN_ID = f"{experiment['name']}-{uuid.uuid4().hex[:8]}"
RUN_DIR = Path('/content/ensemble-runs') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CONFIG_SNAPSHOT = RUN_DIR / 'config.yaml'
CONFIG_SNAPSHOT.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
run = wandb.init(project=experiment['wandb_project'], entity=experiment['wandb_entity'],
    name=RUN_ID, id=RUN_ID, job_type='pretrained-evaluation',
    mode=experiment['wandb_mode'], dir=str(RUN_DIR),
    config={**config, 'github_commit': CODE_COMMIT, 'sam_audio_commit': SAM_COMMIT})
REPORT_PATH = None
MODEL_INFO = None
print('평가 실험:', RUN_ID, run.url)

## 4번 셀 · 사전학습 모델 자동 다운로드 및 로딩
BF16 지원 GPU를 확인하고, 사용하지 않는 영상·구간 예측·재순위 모듈을 제외합니다. 오디오 모델과 가중치는 추가 학습 없이 사용합니다.


In [ ]:
# Set before importing transformers or sam_audio: this evaluation uses PyTorch only.
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'

import gc
import torch
from huggingface_hub import login, snapshot_download
from transformers.utils import is_tf_available
if is_tf_available():
    raise RuntimeError('Transformers가 이전 TensorFlow 설정을 기억하고 있습니다. 세션을 다시 시작하고 1번부터 실행하세요.')
from sam_audio import SAMAudio, SAMAudioProcessor

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError('설정이 변경되었습니다. 3번 셀부터 새 실험을 시작하세요.')
MODEL_INFO = None
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임으로 변경하세요.')
DTYPE = getattr(torch, p['dtype'])
if DTYPE == torch.bfloat16 and torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('이 GPU는 BF16을 기본 지원하지 않습니다. 지원 GPU 또는 float32 설정이 필요합니다.')
hf_token = secret('HF_TOKEN')
if not hf_token:
    raise ValueError('Colab Secrets에 HF_TOKEN을 등록하고 노트북 액세스를 켜세요.')
login(token=hf_token, add_to_git_credential=False)
del hf_token
if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()
MODEL_DIR = Path(snapshot_download(repo_id=p['model_id'], revision=p['model_revision'],
    allow_patterns=['config.json', 'checkpoint.pt']))
processor = SAMAudioProcessor.from_pretrained(str(MODEL_DIR))
# Construct directly in the chosen dtype; mmap avoids a second full checkpoint copy.
model_config = json.loads((MODEL_DIR / 'config.json').read_text())
model_config.update(span_predictor=None, text_ranker=None, visual_ranker=None)
previous_dtype = torch.get_default_dtype()
torch.set_default_dtype(DTYPE)
try:
    model = SAMAudio(SAMAudio.config_cls(**model_config))
finally:
    torch.set_default_dtype(previous_dtype)
state = torch.load(MODEL_DIR / 'checkpoint.pt', weights_only=True, mmap=True, map_location='cpu')
state = {key: value for key, value in state.items() if not key.startswith('vision_encoder.')}
loaded = torch.nn.Module.load_state_dict(model, state, strict=False)
missing = [key for key in loaded.missing_keys if not key.startswith('text_encoder.')]
if missing or loaded.unexpected_keys:
    raise RuntimeError(f'가중치 불일치: {missing}, {loaded.unexpected_keys}')
del state
model = model.eval().requires_grad_(False).to(device='cuda', dtype=DTYPE)
gc.collect()
MODEL_INFO = {'model_id': p['model_id'], 'revision': MODEL_DIR.name,
    'dtype': p['dtype'], 'gpu': torch.cuda.get_device_name(),
    'predict_spans': False, 'reranking_candidates': 1, 'ode_steps': 16,
    'sample_rate': processor.audio_sampling_rate, 'sam_audio_commit': SAM_COMMIT}
MODEL_CONFIG = copy.deepcopy(config)
run.summary['model'] = MODEL_INFO
print(MODEL_INFO)

## 5번 셀 · 악기별 추론과 SDR 평가
전체 음원을 짧은 구간으로 처리합니다. 각 구간 양옆 context를 함께 추론하고 중심 부분만 이어 붙입니다. 이는 공식 long-form multi-diffusion 재현이 아닌 자체 구간 처리 방식이며 경계와 문맥 손실이 점수에 영향을 줄 수 있습니다.
모델 입력은 processor의 샘플레이트로 변환하고 출력은 기존 평가 기준인 44.1kHz mono로 복원합니다. 정답은 추론에 넣지 않습니다.
전체 평가는 오래 걸릴 수 있습니다. max_scenes가 설정되면 전체 점수 대신 diagnostic으로 기록합니다. 오류 후에는 3번 셀부터 다시 실행하세요. 새 실험에서 추론을 처음부터 수행합니다.

In [ ]:
import hashlib
import time
import numpy as np
import soundfile as sf
import torchaudio
from tqdm.auto import tqdm
from ensemble_benchmark.scoring import score_dataset

if (config != yaml.safe_load(CONFIG_SNAPSHOT.read_text())
        or MODEL_INFO is None or MODEL_CONFIG != config):
    raise ValueError('3~4번 셀에서 현재 설정으로 실험과 모델을 준비하세요.')
REPORT_PATH = None
for key in ('evaluation',):
    if key in run.summary.keys():
        del run.summary[key]


def safe_path(root, relative):
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError(f'허용되지 않는 경로: {relative}')
    return path


def separate_audio(audio, rate, prompt, target_key):
    # audio: CPU float32 mono tensor. Reference stems are never passed here.
    n = audio.numel()
    chunk = max(1, round(p['chunk_seconds'] * rate))
    context = round(p['context_seconds'] * rate)
    output = torch.empty(n, dtype=torch.float32)
    for start in tqdm(range(0, n, chunk), desc=target_key, unit='chunk', leave=False):
        end = min(start + chunk, n)
        left, right = max(0, start - context), min(n, end + context)
        segment = torchaudio.functional.resample(audio[left:right].unsqueeze(0),
            rate, processor.audio_sampling_rate)
        # Match the local run's reproducible seed for each scene/target/chunk.
        seed = int.from_bytes(hashlib.sha256(
            f"{p['seed']}/{target_key}/{start}".encode()).digest()[:4], 'big')
        torch.manual_seed(seed)
        batch = processor(audios=[segment], descriptions=[prompt]).to('cuda')
        batch.audios = batch.audios.to(dtype=DTYPE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=DTYPE,
                enabled=DTYPE == torch.bfloat16):
            result = model.separate(batch, predict_spans=False, reranking_candidates=1,
                ode_opt={'method': 'midpoint', 'options': {'step_size': 1 / 16}})
        # Official separate() returns one tensor per batch item in a list.
        target = result.target[0].detach().float().cpu().reshape(-1)
        if target.numel() < segment.shape[-1] or not torch.isfinite(target).all():
            raise ValueError('SAM-Audio 출력 길이가 부족하거나 유한하지 않습니다.')
        target = target[:segment.shape[-1]]
        restored = torchaudio.functional.resample(target, processor.audio_sampling_rate, rate)
        if restored.numel() < right - left:
            raise ValueError('리샘플링 출력 길이가 부족합니다.')
        output[start:end] = restored[start-left:end-left]
        del result, batch
    return output.numpy()


manifest_path = VALID_ROOT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
scenes = manifest['scenes']
if not scenes or any(s.get('split') != 'validation' for s in scenes):
    raise ValueError('validation manifest를 확인하세요.')
if len({s['scene_id'] for s in scenes}) != len(scenes):
    raise ValueError('중복 scene_id입니다.')
if not manifest.get('complete'):
    raise ValueError('완전한 validation manifest가 필요합니다.')
expected = manifest.get('expected_scene_ids')
if not expected or len(set(expected)) != len(expected) or set(expected) != {s['scene_id'] for s in scenes}:
    raise ValueError('validation 장면 목록이 expected_scene_ids와 다릅니다.')
DIAGNOSTIC = p['max_scenes'] is not None
selected = scenes if not DIAGNOSTIC else scenes[:p['max_scenes']]
eval_manifest = copy.deepcopy(manifest)
eval_manifest['scenes'] = selected
eval_manifest['complete'] = not DIAGNOSTIC
PREDICTIONS = RUN_DIR / 'predictions'
for scene in selected:
    for target in scene['target_classes']:
        if target not in p['prompts']:
            raise ValueError(f'YAML에 {target} 프롬프트가 없습니다.')
        safe_path(PREDICTIONS, f"{scene['scene_id']}/{target}.wav")
    if set(scene['references']) != set(scene['target_classes']):
        raise ValueError('정답 악기와 target_classes가 다릅니다.')
    for relative in [scene['mixture'], *scene['references'].values()]:
        path = safe_path(VALID_ROOT, relative)
        info = sf.info(path)
        if (info.samplerate != 44100 or info.channels != 1
                or info.frames != scene['num_samples'] or scene['sample_rate'] != 44100):
            raise ValueError(f'오디오 형식이 manifest와 다릅니다: {relative}')
        with path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != scene['sha256'].get(relative):
            raise ValueError(f'오디오 체크섬 불일치: {relative}')

torch.manual_seed(p['seed'])
np.random.seed(p['seed'])
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
total_targets = sum(len(s['target_classes']) for s in selected)
with tqdm(total=total_targets, desc='SAM-Audio Large', unit='target') as progress:
    for scene in selected:
        audio, rate = sf.read(safe_path(VALID_ROOT, scene['mixture']), dtype='float32')
        tensor = torch.from_numpy(audio)
        for target in scene['target_classes']:
            predicted = separate_audio(tensor, rate, p['prompts'][target], f"{scene['scene_id']}/{target}")
            destination = safe_path(PREDICTIONS, f"{scene['scene_id']}/{target}.wav")
            destination.parent.mkdir(parents=True, exist_ok=True)
            sf.write(destination, predicted, rate, subtype='FLOAT')
            progress.update(1)
            run.log({'inference/completed_targets': progress.n})

torch.cuda.synchronize()
inference_seconds = time.perf_counter() - started
# Release model memory before CPU-based SDR scoring.
del model
gc.collect()
torch.cuda.empty_cache()
report = score_dataset(eval_manifest, VALID_ROOT, PREDICTIONS,
    split='validation', allow_partial=DIAGNOSTIC, show_progress=True)
report.update(diagnostic=DIAGNOSTIC, model=MODEL_INFO, prompts=p['prompts'],
    seed=p['seed'], chunk_seconds=p['chunk_seconds'], context_seconds=p['context_seconds'],
    manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    inference_seconds=inference_seconds,
    peak_gpu_allocated_gb=torch.cuda.max_memory_allocated() / 1024**3)
if DIAGNOSTIC:
    report['overall'] = None
report_path = RUN_DIR / 'evaluation.json'
report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
run.summary['evaluation'] = report
metrics = {'inference/seconds': inference_seconds,
           'inference/peak_gpu_allocated_gb': report['peak_gpu_allocated_gb']}
prefix = 'diagnostic' if DIAGNOSTIC else 'validation'
if report['overall'] is not None:
    metrics[f'{prefix}/sdr'] = report['overall']
for dataset, values in report['datasets'].items():
    for instrument, value in values['instruments'].items():
        metrics[f'{prefix}/{dataset}/{instrument}/sdr'] = value
run.log(metrics)
REPORT_PATH = report_path
print('SDR 평가 완료:', REPORT_PATH)
print('전체 SDR:', report['overall'], '| diagnostic:', DIAGNOSTIC)

## 6번 셀 · W&B 저장 및 결과 ZIP 다운로드
평가 JSON, 설정 YAML, manifest, 환경 목록을 저장합니다. 가중치와 음원은 ZIP에 포함하지 않습니다.


In [ ]:
import subprocess

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError('평가 후 설정이 변경되었습니다. 새 실험을 시작하세요.')
if REPORT_PATH is None or not Path(REPORT_PATH).is_file():
    raise ValueError('먼저 5번 셀의 평가를 완료하세요.')
(RUN_DIR / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
(RUN_DIR / 'evaluation-manifest.json').write_text(json.dumps(eval_manifest, indent=2))
artifact = wandb.Artifact(RUN_ID, type='evaluation', metadata=MODEL_INFO)
for path in (REPORT_PATH, CONFIG_SNAPSHOT, RUN_DIR / 'environment.txt',
             RUN_DIR / 'evaluation-manifest.json'):
    artifact.add_file(str(path), name=path.name)
logged = run.log_artifact(artifact)
if experiment['wandb_mode'] == 'online':
    logged.wait()
run.finish()
print('W&B 평가 결과 저장 완료:', RUN_ID)
# Download a small results bundle; pretrained weights and audio are excluded.
import zipfile
from google.colab import files
bundle = RUN_DIR / f'{RUN_ID}-results.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (REPORT_PATH, CONFIG_SNAPSHOT, RUN_DIR / 'environment.txt',
                 RUN_DIR / 'evaluation-manifest.json'):
        archive.write(path, arcname=path.name)
files.download(str(bundle))
